# Parsing the corpus with MinerU (Google Colab)

This notebook is the only pipeline step that does not run locally (it needs a GPU).

**Input:** chapter PDFs `lay_ch1.pdf … lay_ch5.pdf` in Google Drive, folder `Thesis_GraphRAG/pdf/chapters/` (created by `src/cut_pdf.py`).
**Output:** `Thesis_GraphRAG/parsed/hybrid_full/lay_chN/…/*_content_list.json` plus `images/` for each chapter.

**How to run:** Runtime → Change runtime type → **T4 GPU**, then **Runtime → Run all**.
If the session disconnects, just Run all again: finished chapters are skipped.
If Colab asks to restart the session after installing packages, accept and Run all again.

Pinned for reproducibility: MinerU **3.4.5**, backend **hybrid-auto-engine**, language `en`.

In [ ]:
# 1. Check GPU and Python version (Python 3.10–3.13 required)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Install MinerU (pinned version).
# pip warnings about conflicts with other Colab packages (starlette, huggingface-hub) are harmless.
MINERU_VERSION = "3.4.5"
!pip install -q "mineru[core]=={MINERU_VERSION}"
!mineru --version

In [ ]:
# 4. Paths and parameters
import os, glob, time

BASE = "/content/drive/MyDrive/Thesis_GraphRAG"
CH_DIR = f"{BASE}/pdf/chapters"
OUT_DIR = f"{BASE}/parsed/hybrid_full"
BACKEND = "hybrid-auto-engine"
os.makedirs(OUT_DIR, exist_ok=True)

chapters = sorted(glob.glob(f"{CH_DIR}/lay_ch*.pdf"))
assert chapters, f"No chapter PDFs found in {CH_DIR}. Upload the files from data/pdf/chapters/"
print("Chapters:", [os.path.basename(c) for c in chapters])

In [ ]:
# 5. Parse (finished chapters are skipped)
timings = {}
for pdf in chapters:
    name = os.path.splitext(os.path.basename(pdf))[0]          # lay_ch1
    if glob.glob(f"{OUT_DIR}/{name}/**/*_content_list.json", recursive=True):
        print(f"✅ {name}: already parsed, skipping")
        continue
    print(f"⏳ {name}: parsing...")
    t0 = time.time()
    !mineru -p "{pdf}" -o "{OUT_DIR}" -b {BACKEND} -d cuda -m auto -l en
    timings[name] = round((time.time() - t0) / 60, 1)
    print(f"✅ {name}: {timings[name]} min")
print("Parsing time (min):", timings)

In [ ]:
# 6. Check the output
import json
for pdf in chapters:
    name = os.path.splitext(os.path.basename(pdf))[0]
    files = [f for f in glob.glob(f"{OUT_DIR}/{name}/**/*_content_list.json", recursive=True) if not f.endswith("_v2.json")]
    if not files:
        print(f"❌ {name}: no content_list"); continue
    items = json.load(open(files[0], encoding="utf-8"))
    types = {}
    for it in items:
        types[it.get("type")] = types.get(it.get("type"), 0) + 1
    print(f"{name}: {len(items)} blocks {types}")
print("\nNext: download parsed/hybrid_full from Drive into data/parsed/hybrid_full locally (see README).")